In [1]:
import requests

# Decoding 키 사용 (requests가 알아서 인코딩해줌)
API_KEY = "6lVhhlLRaGq/+tidZgS0POWCcl7BOqRJXiDj+Xtl/+rJJVEqNPWFjwFpyWkOn3NaNqacOHvj9UG+BnHAGBFd4w=="

url = 'https://apis.data.go.kr/9760000/ErVotingSttusInfoInqireService/getErVotingSttusInfoInqire'
params = {
    'serviceKey': API_KEY,
    'pageNo': '1',
    'numOfRows': '10',
    'resultType': 'json',
    'sgId': '20231011',
    'erVotingDiv': '1',
    'sdName': '서울특별시',
    'wiwName': '강서구'
}

response = requests.get(url, params=params)
print(response.status_code)
print(response.json())

200
{'response': {'header': {'resultCode': 'INFO-00', 'resultMsg': 'NORMAL SERVICE'}, 'body': {'items': {'item': [{'num': '1', 'sgId': '20231011', 'erVotingDiv': '1', 'sdName': '서울특별시', 'wiwName': '강서구', 'votersCnt': '500603', 'erVotingCnt': '42429', 'erTurnout': '8.48', 'sortOrd': '3'}]}, 'numOfRows': 10, 'pageNo': 1, 'totalCount': 1}}}


In [5]:
import requests
import pandas as pd
import time

# ============================================================
# 설정
# ============================================================
API_KEY = "6lVhhlLRaGq/+tidZgS0POWCcl7BOqRJXiDj+Xtl/+rJJVEqNPWFjwFpyWkOn3NaNqacOHvj9UG+BnHAGBFd4w=="

BASE_URL = "https://apis.data.go.kr/9760000/ErVotingSttusInfoInqireService/getErVotingSttusInfoInqire"

# 비교할 선거 목록 (sgId: 선거일자 형식)
ELECTIONS = {
    "20200415": "제21대 국회의원선거",
    "20220309": "제20대 대통령선거",
    "20220601": "제8회 전국동시지방선거",
    "20231011": "제22대 국선 강서구청장 보궐선거",
    "20240410": "제22대 국회의원선거",
    "20250603": "제21대 대통령선거",
}

# 사전투표 구분 (0=전체합산, 1=1일차, 2=2일차)
ER_VOTING_DIV = "0"  # 전체 합산 기준


# ============================================================
# 데이터 수집 함수
# ============================================================
def fetch_all_pages(sg_id, er_voting_div, num_of_rows=100):
    """한 선거의 전체 페이지 데이터를 수집"""
    all_items = []
    page = 1

    while True:
        params = {
            "serviceKey": API_KEY,
            "pageNo": str(page),
            "numOfRows": str(num_of_rows),
            "resultType": "json",
            "sgId": sg_id,
            "erVotingDiv": er_voting_div,
        }

        try:
            response = requests.get(BASE_URL, params=params, timeout=10)
            data = response.json()

            header = data["response"]["header"]
            if header["resultCode"] != "INFO-00":
                print(f"  ⚠️ API 오류: {header['resultMsg']}")
                break

            body = data["response"]["body"]
            total_count = body["totalCount"]
            items_data = body.get("items", {})

            # 데이터가 없는 경우
            if not items_data or not items_data.get("item"):
                print(f"  ℹ️ 데이터 없음 (sgId={sg_id})")
                break

            items = items_data["item"]
            # 단일 결과인 경우 리스트로 변환
            if isinstance(items, dict):
                items = [items]

            all_items.extend(items)
            print(f"  페이지 {page} 수집: {len(items)}건 (누적: {len(all_items)}/{total_count})")

            # 마지막 페이지 확인
            if len(all_items) >= int(total_count):
                break

            page += 1
            time.sleep(0.3)  # API 서버 부하 방지

        except Exception as e:
            print(f"  ❌ 오류 발생 (페이지 {page}): {e}")
            break

    return all_items


# ============================================================
# 전체 선거 데이터 수집
# ============================================================
all_records = []

for sg_id, election_name in ELECTIONS.items():
    print(f"\n{'='*50}")
    print(f"📌 수집 중: {election_name} (sgId={sg_id})")
    print(f"{'='*50}")

    items = fetch_all_pages(sg_id, ER_VOTING_DIV)

    for item in items:
        item["election_name"] = election_name  # 선거명 컬럼 추가

    all_records.extend(items)
    print(f"  ✅ 완료: {len(items)}건")
    time.sleep(0.5)


# ============================================================
# DataFrame 변환 및 정리
# ============================================================
df = pd.DataFrame(all_records)

# 컬럼 순서 및 한글명 정리
column_map = {
    "election_name": "선거명",
    "sgId":          "선거ID",
    "erVotingDiv":   "사전투표구분",
    "sdName":        "시도명",
    "wiwName":       "구시군명",
    "votersCnt":     "선거인수",
    "erVotingCnt":   "사전투표자수",
    "erTurnout":     "사전투표율(%)",
    "sortOrd":       "정렬순서",
    "num":           "결과순서",
}

# 존재하는 컬럼만 선택 후 이름 변경
existing_cols = [c for c in column_map if c in df.columns]
df = df[existing_cols].rename(columns=column_map)

# 숫자형 변환
for col in ["선거인수", "사전투표자수", "사전투표율(%)"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 사전투표구분 값 → 텍스트
div_map = {"0": "전체", "1": "1일차", "2": "2일차"}
if "사전투표구분" in df.columns:
    df["사전투표구분"] = df["사전투표구분"].map(div_map).fillna(df["사전투표구분"])

print(f"\n\n{'='*50}")
print(f"✅ 총 수집 완료: {len(df)}건")
print(df.head(10).to_string(index=False))


# ============================================================
# CSV 저장
# ============================================================
output_path = "early_voting_data.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")  # utf-8-sig: 엑셀 한글 깨짐 방지
print(f"\n💾 저장 완료: {output_path}")
print(f"   총 {len(df)}행 × {len(df.columns)}열")


📌 수집 중: 제21대 국회의원선거 (sgId=20200415)
  페이지 1 수집: 100건 (누적: 100/268)
  페이지 2 수집: 100건 (누적: 200/268)
  페이지 3 수집: 68건 (누적: 268/268)
  ✅ 완료: 268건

📌 수집 중: 제20대 대통령선거 (sgId=20220309)
  페이지 1 수집: 100건 (누적: 100/268)
  페이지 2 수집: 100건 (누적: 200/268)
  페이지 3 수집: 68건 (누적: 268/268)
  ✅ 완료: 268건

📌 수집 중: 제8회 전국동시지방선거 (sgId=20220601)
  페이지 1 수집: 100건 (누적: 100/268)
  페이지 2 수집: 100건 (누적: 200/268)
  페이지 3 수집: 68건 (누적: 268/268)
  ✅ 완료: 268건

📌 수집 중: 제22대 국선 강서구청장 보궐선거 (sgId=20231011)
  페이지 1 수집: 3건 (누적: 3/3)
  ✅ 완료: 3건

📌 수집 중: 제22대 국회의원선거 (sgId=20240410)
  페이지 1 수집: 100건 (누적: 100/271)
  페이지 2 수집: 100건 (누적: 200/271)
  페이지 3 수집: 71건 (누적: 271/271)
  ✅ 완료: 271건

📌 수집 중: 제21대 대통령선거 (sgId=20250603)
  페이지 1 수집: 100건 (누적: 100/271)
  페이지 2 수집: 100건 (누적: 200/271)
  페이지 3 수집: 71건 (누적: 271/271)
  ✅ 완료: 271건


✅ 총 수집 완료: 1349건
        선거명     선거ID 사전투표구분   시도명 구시군명     선거인수   사전투표자수  사전투표율(%) 정렬순서 결과순서
제21대 국회의원선거 20200415     전체    합계   합계 43994247 11742677     26.69    1    1
제21대 국회의원선거 20200415     전체 서울특별시   합계